# **FRAMEWORK V7 - EVALUACIÓN DE MODELOS**


## **M0. Configuración del Experimento de Predicción**

In [1]:
#==========================================================================================
# SCRIPT 50
# CONFIGURACIÓN DEL EXPERIMENTO DE PREDICCIÓN
#==========================================================================================

EXPERIMENTO = "Exp01"

print()

print("="*90)
print("CONFIGURACIÓN DEL EXPERIMENTO")
print("="*90)

print()

print(f"Experimento seleccionado : {EXPERIMENTO}")


CONFIGURACIÓN DEL EXPERIMENTO

Experimento seleccionado : Exp01


## **M1. Librerias**

In [2]:
#==========================================================================================
# SCRIPT 51
# LIBRERÍAS
#==========================================================================================

import numpy as np

import pandas as pd

import tensorflow as tf

from tensorflow import keras

print()

print("="*90)
print("LIBRERÍAS CARGADAS")
print("="*90)


LIBRERÍAS CARGADAS


## **M2. Carga Automática del Modelo y la Metadata**

In [3]:
#==========================================================================================
# SCRIPT 52
# CARGA AUTOMÁTICA DEL MODELO Y LA METADATA
#==========================================================================================

import pandas as pd
import tensorflow as tf
from tensorflow import keras

#------------------------------------------------------------------------------
# RUTAS DEL FRAMEWORK
#------------------------------------------------------------------------------

BASE_URL = (
    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/refs/heads/main/DATA/MODELADO/"
)

BASE_MODELOS = BASE_URL + "Modelos/"

BASE_TENSORES = BASE_URL + "Tensores/"

#------------------------------------------------------------------------------
# CARGA DE LA METADATA
#------------------------------------------------------------------------------

metadata_url = (

    BASE_TENSORES
    + EXPERIMENTO
    + "/metadata_tensor.csv"

)

!wget -q -O metadata_tensor.csv {metadata_url}

metadata = pd.read_csv(

    "metadata_tensor.csv"

)

#------------------------------------------------------------------------------
# RECUPERAR CONFIGURACIÓN DEL EXPERIMENTO
#------------------------------------------------------------------------------

DOMINIO = metadata.loc[
    metadata["Parametro"] == "Dominio",
    "Valor"
].iloc[0]

VARIABLE_OBJETIVO = metadata.loc[
    metadata["Parametro"] == "Variable Objetivo",
    "Valor"
].iloc[0]

VENTANA = int(
    metadata.loc[
        metadata["Parametro"] == "Ventana",
        "Valor"
    ].iloc[0]
)

HORIZONTE = int(
    metadata.loc[
        metadata["Parametro"] == "Horizonte",
        "Valor"
    ].iloc[0]
)

NUM_VARIABLES = int(
    metadata.loc[
        metadata["Parametro"] == "Variables Predictoras",
        "Valor"
    ].iloc[0]
)

METODO_TRANSFORMACION = metadata.loc[
    metadata["Parametro"] == "Metodo Transformacion",
    "Valor"
].iloc[0]

NUM_SECUENCIAS = int(
    metadata.loc[
        metadata["Parametro"] == "Numero Secuencias",
        "Valor"
    ].iloc[0]
)

#------------------------------------------------------------------------------
# CONSTRUIR AUTOMÁTICAMENTE EL NOMBRE DEL MODELO
#------------------------------------------------------------------------------

modelo_url = (

    BASE_MODELOS
    + EXPERIMENTO
    + "/modelo_"
    + EXPERIMENTO
    + "_"
    + VARIABLE_OBJETIVO
    + ".keras"

)

#------------------------------------------------------------------------------
# DESCARGA DEL MODELO
#------------------------------------------------------------------------------

!wget -q -O modelo.keras {modelo_url}

#------------------------------------------------------------------------------
# CARGA DEL MODELO
#------------------------------------------------------------------------------

modelo = keras.models.load_model(

    "modelo.keras"

)

#------------------------------------------------------------------------------
# RESUMEN
#------------------------------------------------------------------------------

print()

print("="*90)
print("MODELO Y METADATA CARGADOS")
print("="*90)

print()

print(f"Experimento            : {EXPERIMENTO}")
print(f"Dominio                : {DOMINIO}")
print(f"Variable objetivo      : {VARIABLE_OBJETIVO}")
print(f"Ventana                : {VENTANA}")
print(f"Horizonte              : {HORIZONTE}")
print(f"Variables predictoras  : {NUM_VARIABLES}")
print(f"Método transformación  : {METODO_TRANSFORMACION}")
print(f"Número de secuencias   : {NUM_SECUENCIAS}")

print()

print("Modelo cargado correctamente.")

print()

display(metadata)


MODELO Y METADATA CARGADOS

Experimento            : Exp01
Dominio                : Gestión Hídrica
Variable objetivo      : irca
Ventana                : 12
Horizonte              : 1
Variables predictoras  : 8
Método transformación  : Escalado MinMax
Número de secuencias   : 516

Modelo cargado correctamente.



,Parametro,Valor
0,Experimento,Exp01
1,Dominio,Gestión Hídrica
2,Variable Objetivo,irca
3,Ventana,12
4,Horizonte,1
5,Variables Predictoras,8
6,Variables Modelo,Precipitacion_mm;ONI;Radiacion_Solar;Humedad_R...
7,Metodo Transformacion,Escalado MinMax
8,Numero Secuencias,516
9,Fecha Generacion,2026-07-28 16:08


## **M3. Carga, validación y preparación del dataset.**

In [4]:
#==========================================================================================
# SCRIPT 53
# CARGA DEL DATASET PARA PREDICCIÓN
#==========================================================================================

URL_DATASET = (

    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/refs/heads/main/"
    "DATA/MACHINE_LEARNING/C13_MACHINE_LEARNING/Transformaciones/"
    + EXPERIMENTO +
    "/dataset_machine_learning_transformado.parquet"

)

dataset_prediccion = pd.read_parquet(

    URL_DATASET

)

print()

print("="*90)
print("DATASET PARA PREDICCIÓN")
print("="*90)

print()

print(f"Observaciones : {dataset_prediccion.shape[0]}")

print(f"Variables     : {dataset_prediccion.shape[1]}")

print()

display(dataset_prediccion.head())


DATASET PARA PREDICCIÓN

Observaciones : 528
Variables     : 9



,Precipitacion_mm,ONI,Radiacion_Solar,Humedad_Relativa,Temp_Min_C,Mes,Velocidad_Viento,VolumenUtilDiarioMasa,irca
0,-0.594799,0.570842,0.628362,-1.191977,-0.710044,-1.000000,0.944444,0.0,0
1,-0.685579,0.480493,0.325183,-1.055874,0.621605,-0.818182,0.305556,0.0,0
2,-0.112530,0.480493,0.066015,-0.848138,0.571068,-0.636364,0.166667,0.0,0
3,0.059574,0.685832,0.031785,-0.842407,0.391661,-0.454545,0.111111,0.0,0
4,-0.486998,0.858316,-0.242054,-0.949857,0.247631,-0.272727,0.305556,0.0,0


In [5]:

#==========================================================================================
# VALIDACIÓN DEL DATASET
#==========================================================================================

VARIABLES_MODELO = metadata.loc[
    metadata["Parametro"] == "Variables Modelo",
    "Valor"
].iloc[0].split(";")

print()

print("="*90)
print("VALIDACIÓN DEL DATASET")
print("="*90)

print()

print(f"Variables esperadas : {len(VARIABLES_MODELO)}")
print(f"Variables encontradas : {dataset_prediccion.shape[1]}")

#------------------------------------------------------------------------------
# VALIDACIÓN DE VARIABLES
#------------------------------------------------------------------------------

faltantes = [

    variable

    for variable in VARIABLES_MODELO

    if variable not in dataset_prediccion.columns

]

if len(faltantes) > 0:

    raise Exception(

        "El dataset no contiene las siguientes variables:\n\n"

        + "\n".join(faltantes)

    )

print()

print("Resultado : VARIABLES VÁLIDAS")

print()

print("Variables utilizadas por el modelo:")

for variable in VARIABLES_MODELO:

    print(f"• {variable}")


VALIDACIÓN DEL DATASET

Variables esperadas : 8
Variables encontradas : 9

Resultado : VARIABLES VÁLIDAS

Variables utilizadas por el modelo:
• Precipitacion_mm
• ONI
• Radiacion_Solar
• Humedad_Relativa
• Temp_Min_C
• Mes
• Velocidad_Viento
• VolumenUtilDiarioMasa


## **M4. Preparación del Dataset para el Modelo**

In [6]:
#==========================================================================================
# SCRIPT 54
# PREPARACIÓN DEL DATASET PARA EL MODELO
#==========================================================================================

# ==========================================================================================
#
# PREPARACIÓN DEL DATASET PARA EL MODELO
#
# Observación
#
# En esta etapa el Framework prepara automáticamente el conjunto de datos que será
# utilizado por el modelo de predicción.
#
# La selección de variables no depende del usuario, sino de la metadata generada
# durante el proceso de entrenamiento (metadata_tensor.csv), garantizando que las
# variables predictoras correspondan exactamente con aquellas utilizadas durante la
# construcción y entrenamiento del modelo.
#
# Este mecanismo asegura la consistencia entre entrenamiento y predicción,
# eliminando errores ocasionados por diferencias en el orden, cantidad o nombre de
# las variables de entrada.
#==========================================================================================


dataset_modelo = dataset_prediccion[

    VARIABLES_MODELO

].copy()

print()

print("="*90)
print("DATASET PREPARADO PARA EL MODELO")
print("="*90)

print()

print(f"Observaciones : {dataset_modelo.shape[0]}")

print(f"Variables     : {dataset_modelo.shape[1]}")

print()

display(dataset_modelo.head())


DATASET PREPARADO PARA EL MODELO

Observaciones : 528
Variables     : 8



,Precipitacion_mm,ONI,Radiacion_Solar,Humedad_Relativa,Temp_Min_C,Mes,Velocidad_Viento,VolumenUtilDiarioMasa
0,-0.594799,0.570842,0.628362,-1.191977,-0.710044,-1.000000,0.944444,0.0
1,-0.685579,0.480493,0.325183,-1.055874,0.621605,-0.818182,0.305556,0.0
2,-0.112530,0.480493,0.066015,-0.848138,0.571068,-0.636364,0.166667,0.0
3,0.059574,0.685832,0.031785,-0.842407,0.391661,-0.454545,0.111111,0.0
4,-0.486998,0.858316,-0.242054,-0.949857,0.247631,-0.272727,0.305556,0.0


In [7]:

dataset_modelo = dataset_prediccion[

    VARIABLES_MODELO

].copy()

print()

print("="*90)
print("DATASET PREPARADO PARA EL MODELO")
print("="*90)

print()

print(f"Observaciones : {dataset_modelo.shape[0]}")

print(f"Variables     : {dataset_modelo.shape[1]}")

print()

display(dataset_modelo.head())


DATASET PREPARADO PARA EL MODELO

Observaciones : 528
Variables     : 8



,Precipitacion_mm,ONI,Radiacion_Solar,Humedad_Relativa,Temp_Min_C,Mes,Velocidad_Viento,VolumenUtilDiarioMasa
0,-0.594799,0.570842,0.628362,-1.191977,-0.710044,-1.000000,0.944444,0.0
1,-0.685579,0.480493,0.325183,-1.055874,0.621605,-0.818182,0.305556,0.0
2,-0.112530,0.480493,0.066015,-0.848138,0.571068,-0.636364,0.166667,0.0
3,0.059574,0.685832,0.031785,-0.842407,0.391661,-0.454545,0.111111,0.0
4,-0.486998,0.858316,-0.242054,-0.949857,0.247631,-0.272727,0.305556,0.0


## **M5. Construcción de Secuencias para Predicción**

In [8]:
#==========================================================================================
# SCRIPT 55
# CONSTRUCCIÓN DE SECUENCIAS PARA PREDICCIÓN
#
# Observación
#
# El Framework construye automáticamente las secuencias temporales utilizando la
# misma longitud de ventana empleada durante el entrenamiento del modelo.
#
# Esta estrategia garantiza que la estructura de entrada utilizada durante la
# predicción sea idéntica a la utilizada durante el proceso de aprendizaje,
# preservando la consistencia temporal requerida por las redes neuronales LSTM.
#
# Durante esta etapa únicamente se generan las secuencias de entrada (X), ya que
# la variable objetivo es desconocida y será estimada posteriormente por el modelo
# entrenado.
#
#==========================================================================================

# Contenedor de secuencias
secuencias_prediccion = []

# Construcción de ventanas temporales
for i in range(

    len(dataset_modelo) - VENTANA + 1

):

    secuencia = dataset_modelo.iloc[

        i : i + VENTANA

    ].values

    secuencias_prediccion.append(

        secuencia

    )

print()

print("="*90)
print("SECUENCIAS DE PREDICCIÓN")
print("="*90)

print()

print(f"Ventana temporal      : {VENTANA}")

print(f"Variables predictoras : {dataset_modelo.shape[1]}")

print(f"Secuencias generadas  : {len(secuencias_prediccion)}")


SECUENCIAS DE PREDICCIÓN

Ventana temporal      : 12
Variables predictoras : 8
Secuencias generadas  : 517


In [9]:
#==========================================================================================
# PRIMERA SECUENCIA
#==========================================================================================

primera_secuencia = pd.DataFrame(

    secuencias_prediccion[0],

    columns=dataset_modelo.columns

)

print()

print("="*90)
print("PRIMERA SECUENCIA")
print("="*90)

print()

display(primera_secuencia)


PRIMERA SECUENCIA



,Precipitacion_mm,ONI,Radiacion_Solar,Humedad_Relativa,Temp_Min_C,Mes,Velocidad_Viento,VolumenUtilDiarioMasa
0,-0.594799,0.570842,0.628362,-1.191977,-0.710044,-1.000000,0.944444,0.0
1,-0.685579,0.480493,0.325183,-1.055874,0.621605,-0.818182,0.305556,0.0
2,-0.112530,0.480493,0.066015,-0.848138,0.571068,-0.636364,0.166667,0.0
3,0.059574,0.685832,0.031785,-0.842407,0.391661,-0.454545,0.111111,0.0
4,-0.486998,0.858316,-0.242054,-0.949857,0.247631,-0.272727,0.305556,0.0
5,-0.341371,1.022587,-0.574572,-1.293696,0.186987,-0.090909,2.361111,0.0
6,-0.343262,1.244353,0.036675,-1.644699,-0.048010,0.090909,2.000000,0.0
7,-0.623168,1.630390,-0.168704,-2.160458,0.138977,0.272727,1.888889,0.0
8,-0.630733,1.852156,0.985330,-2.704871,-0.234997,0.454545,1.666667,0.0
9,-0.254374,1.991786,0.408313,-1.906877,0.419457,0.636364,0.166667,0.0


## **M6. Construcción del Tensor de Predicción**

In [10]:
#==========================================================================================
# SCRIPT 56
# CONSTRUCCIÓN DEL TENSOR DE PREDICCIÓN
#==========================================================================================

X_pred = np.array(

    secuencias_prediccion,

    dtype=np.float32

)

print()

print("="*90)
print("TENSOR DE PREDICCIÓN")
print("="*90)

print()

print(f"Forma : {X_pred.shape}")


TENSOR DE PREDICCIÓN

Forma : (517, 12, 8)


In [11]:
#==========================================================================================
# VALIDACIÓN DEL TENSOR
#
# Observación
#
# En esta etapa el Framework convierte automáticamente las secuencias temporales
# en un tensor tridimensional compatible con la arquitectura LSTM.
#
# El tensor generado conserva exactamente la misma estructura utilizada durante el
# entrenamiento del modelo:
#
# (número de secuencias, ventana temporal, variables predictoras)
#
# Antes de ejecutar la inferencia, el Framework valida que la dimensión temporal
# coincida con la ventana utilizada durante el entrenamiento, garantizando la
# compatibilidad entre los datos de entrada y el modelo cargado.
#
#==========================================================================================

print()

print("="*90)
print("VALIDACIÓN")
print("="*90)

print()

print(f"Tensor X : {X_pred.shape}")

print()

print(f"Ventana temporal      : {X_pred.shape[1]}")

print(f"Variables predictoras : {X_pred.shape[2]}")

print()

if X_pred.shape[1] == VENTANA:

    print("Resultado : TENSOR VÁLIDO")

else:

    raise Exception(

        "La ventana temporal del tensor no coincide con la del modelo."

    )


VALIDACIÓN

Tensor X : (517, 12, 8)

Ventana temporal      : 12
Variables predictoras : 8

Resultado : TENSOR VÁLIDO


## **M7. Predicción del Modelo**

In [12]:
#==========================================================================================
# SCRIPT 57
# PREDICCIÓN DEL MODELO
#==========================================================================================

predicciones = modelo.predict(

    X_pred,

    verbose=0

)

print()

print("="*90)
print("PREDICCIÓN DEL MODELO")
print("="*90)

print()

print(f"Predicciones generadas : {len(predicciones)}")


PREDICCIÓN DEL MODELO

Predicciones generadas : 517


In [13]:
#==========================================================================================
# RESULTADO DE LAS PREDICCIONES
#==========================================================================================

predicciones_df = pd.DataFrame({

    "Prediccion": predicciones.flatten()

})

print()

print("="*90)
print("RESULTADOS")
print("="*90)

print()

display(predicciones_df.head())


RESULTADOS



,Prediccion
0,-1.469519
1,-1.451446
2,-1.573496
3,-1.626176
4,-1.691092


In [14]:
#==========================================================================================
# RESUMEN ESTADÍSTICO
#==========================================================================================

print()

print("="*90)
print("RESUMEN ESTADÍSTICO")
print("="*90)

print()

display(

    predicciones_df.describe().T

)


RESUMEN ESTADÍSTICO



,count,mean,std,min,25%,50%,75%,max
Prediccion,517.0,-0.70697,0.537433,-1.691092,-1.048795,-0.802764,-0.447604,2.090697


## **M8. Consolidación de Resultados**

In [15]:
#==========================================================================================
# SCRIPT 58
# CONSOLIDACIÓN DE RESULTADOS
#
# Observación
#
# El Framework consolida las predicciones generadas por el modelo en una estructura
# tabular que facilita su análisis, almacenamiento y posterior integración con
# otros procesos.
#
# Cada fila representa la estimación obtenida para una secuencia temporal
# construida automáticamente a partir del conjunto de datos suministrado por el
# usuario.
#
# Esta estructura constituye el resultado principal del proceso de inferencia y
# servirá como base para la exportación y visualización de las predicciones.
#==========================================================================================


resultados = pd.DataFrame({

    "Registro": range(

        VENTANA,

        VENTANA + len(predicciones)

    ),

    "Prediccion": predicciones.flatten()

})

print()

print("="*90)
print("RESULTADOS CONSOLIDADOS")
print("="*90)

print()

print(f"Predicciones generadas : {len(resultados)}")

print()

display(resultados.head(10))


RESULTADOS CONSOLIDADOS

Predicciones generadas : 517



,Registro,Prediccion
0,12,-1.469519
1,13,-1.451446
2,14,-1.573496
3,15,-1.626176
4,16,-1.691092
5,17,-1.627071
6,18,-1.610258
7,19,-1.636854
8,20,-1.616144
9,21,-1.596102


In [17]:
#==========================================================================================
# SCRIPT 52A
# CARGA DEL SCALER
#
# Transformador recuperado correctamente.
#
# El scaler será utilizado para transformar automáticamente las variables
# predictoras antes de ejecutar la inferencia del modelo.
#
# La variable objetivo no requiere transformación inversa, ya que durante el
# entrenamiento fue conservada en su escala original.
#
#==========================================================================================

import joblib

#------------------------------------------------------------------------------
# URL DEL SCALER
#------------------------------------------------------------------------------

BASE_URL1 = (
    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/refs/heads/main/DATA/"
)

BASE_SCALER = BASE_URL1 + "MACHINE_LEARNING/C13_MACHINE_LEARNING/Transformaciones/"


scaler_url = (

    BASE_SCALER
    + EXPERIMENTO
    + "/scaler.pkl"

)

#------------------------------------------------------------------------------
# DESCARGA
#------------------------------------------------------------------------------

!wget -q -O scaler.pkl {scaler_url}

#------------------------------------------------------------------------------
# CARGA
#------------------------------------------------------------------------------

scaler = joblib.load(

    "scaler.pkl"

)

print()

print("="*90)
print("SCALER CARGADO")
print("="*90)

print()

print("Transformador recuperado correctamente.")
print()

print(type(scaler))


SCALER CARGADO

Transformador recuperado correctamente.

<class 'sklearn.preprocessing._data.RobustScaler'>


## **M9. Exportación de Las Predicciones**

In [18]:
#==========================================================================================
# SCRIPT 59
# EXPORTACIÓN DE LAS PREDICCIONES
#==========================================================================================

import os

CARPETA_PREDICCIONES = f"predicciones/{EXPERIMENTO}"

os.makedirs(

    CARPETA_PREDICCIONES,

    exist_ok=True

)

#------------------------------------------------------------------------------
# CSV
#------------------------------------------------------------------------------

resultados.to_csv(

    f"{CARPETA_PREDICCIONES}/predicciones.csv",

    index=False,

    encoding="utf-8-sig"

)

#------------------------------------------------------------------------------
# EXCEL
#------------------------------------------------------------------------------

resultados.to_excel(

    f"{CARPETA_PREDICCIONES}/predicciones.xlsx",

    index=False

)

print()

print("="*90)
print("PREDICCIONES EXPORTADAS")
print("="*90)

print()

print(f"{CARPETA_PREDICCIONES}/predicciones.csv")

print(f"{CARPETA_PREDICCIONES}/predicciones.xlsx")


PREDICCIONES EXPORTADAS

predicciones/Exp01/predicciones.csv
predicciones/Exp01/predicciones.xlsx


## **M10. Metadata de la Predicción**

In [19]:
#==========================================================================================
# SCRIPT 60
# METADATA DE LA PREDICCIÓN
#==========================================================================================

metadata_prediccion = pd.DataFrame({

    "Parametro":[

        "Experimento",

        "Dominio",

        "Variable Objetivo",

        "Modelo",

        "Metodo Transformacion",

        "Ventana",

        "Variables Predictoras",

        "Registros Procesados",

        "Predicciones Generadas",

        "Fecha Ejecucion"

    ],

    "Valor":[

        EXPERIMENTO,

        DOMINIO,

        VARIABLE_OBJETIVO,

        f"modelo_{EXPERIMENTO}_{VARIABLE_OBJETIVO}.keras",

        METODO_TRANSFORMACION,

        VENTANA,

        NUM_VARIABLES,

        len(dataset_prediccion),

        len(resultados),

        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

    ]

})

print()

print("="*90)
print("METADATA DE LA PREDICCIÓN")
print("="*90)

print()

display(metadata_prediccion)


METADATA DE LA PREDICCIÓN



,Parametro,Valor
0,Experimento,Exp01
1,Dominio,Gestión Hídrica
2,Variable Objetivo,irca
3,Modelo,modelo_Exp01_irca.keras
4,Metodo Transformacion,Escalado MinMax
5,Ventana,12
6,Variables Predictoras,8
7,Registros Procesados,528
8,Predicciones Generadas,517
9,Fecha Ejecucion,2026-07-28 17:06


In [20]:
#==========================================================================================
# EXPORTACIÓN DE LA METADATA
#==========================================================================================

metadata_prediccion.to_csv(

    f"{CARPETA_PREDICCIONES}/metadata_prediccion.csv",

    index=False,

    encoding="utf-8-sig"

)

metadata_prediccion.to_excel(

    f"{CARPETA_PREDICCIONES}/metadata_prediccion.xlsx",

    index=False

)

print()

print("="*90)
print("METADATA EXPORTADA")
print("="*90)

print()

print(f"{CARPETA_PREDICCIONES}/metadata_prediccion.csv")

print(f"{CARPETA_PREDICCIONES}/metadata_prediccion.xlsx")


METADATA EXPORTADA

predicciones/Exp01/metadata_prediccion.csv
predicciones/Exp01/metadata_prediccion.xlsx


## **M11. Resumen Final del Framework**

In [21]:
#==========================================================================================
# SCRIPT 61
# RESUMEN FINAL DEL FRAMEWORK
#==========================================================================================

print()

print("="*90)
print("FRAMEWORK DE PREDICCIÓN FINALIZADO")
print("="*90)

print()

print(f"Experimento              : {EXPERIMENTO}")

print(f"Dominio                  : {DOMINIO}")

print(f"Variable objetivo        : {VARIABLE_OBJETIVO}")

print()

print(f"Modelo utilizado         : modelo_{EXPERIMENTO}_{VARIABLE_OBJETIVO}.keras")

print(f"Método transformación    : {METODO_TRANSFORMACION}")

print()

print(f"Registros procesados     : {len(dataset_prediccion)}")

print(f"Predicciones generadas   : {len(resultados)}")

print()

print("Archivos generados")

print("------------------------------")

print("✓ predicciones.csv")

print("✓ predicciones.xlsx")

print("✓ metadata_prediccion.csv")

print("✓ metadata_prediccion.xlsx")

print()

print("Proceso finalizado correctamente.")


FRAMEWORK DE PREDICCIÓN FINALIZADO

Experimento              : Exp01
Dominio                  : Gestión Hídrica
Variable objetivo        : irca

Modelo utilizado         : modelo_Exp01_irca.keras
Método transformación    : Escalado MinMax

Registros procesados     : 528
Predicciones generadas   : 517

Archivos generados
------------------------------
✓ predicciones.csv
✓ predicciones.xlsx
✓ metadata_prediccion.csv
✓ metadata_prediccion.xlsx

Proceso finalizado correctamente.
